# Reach the declared state object through a lifecycle

Paid describes where an order stands in a process. The lifecycle defines how an order can reach it; the state object supplies its code, label and additional data.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
from pydantic import Field

from aoa.action_machine.domain import BaseStateEntity, Lifecycle, value
from aoa.action_machine.graph.core.exclude_graph_model import excluded_from_graph_model


## Declare the model

In [ ]:
class OrderStateEntity(BaseStateEntity):
    """Describe a possible order state."""

    requires_receipt: bool = Field(default=False, description="Whether a receipt is required")

    DRAFT = value(code="draft", display_name="Draft")
    PAID = value(code="paid", display_name="Payment received", requires_receipt=True)



class OrderLifecycle(Lifecycle[OrderStateEntity]):
    """Allow an order to move from draft to paid."""

    _template = Lifecycle().state(OrderStateEntity.DRAFT).to(OrderStateEntity.PAID).initial().state(OrderStateEntity.PAID).final()

paid = OrderLifecycle("draft").transition("paid")
assert paid.current_status is OrderStateEntity.PAID
assert excluded_from_graph_model(OrderStateEntity)
print("code:", paid.current_state)
print("receipt required:", paid.current_status.requires_receipt)
print("shared declared object:", paid.current_status is OrderStateEntity.PAID)
print("excluded from entity graph:", excluded_from_graph_model(OrderStateEntity))

## Run this case

In [3]:
paid = OrderLifecycle("draft").transition("paid")
assert paid.current_status is OrderLifecycle.PAID
assert excluded_from_graph_model(OrderStateEntity)
print("code:", paid.current_state)
print("receipt required:", paid.current_status.requires_receipt)
print("shared declared object:", paid.current_status is OrderLifecycle.PAID)
print("excluded from entity graph:", excluded_from_graph_model(OrderStateEntity))


code: paid
receipt required: True
shared declared object: True
excluded from entity graph: True


State supplies code and display_name and validates that neither is blank. It is immutable and excluded from the ordinary Entity graph. The lifecycle can still export its states as State nodes. A state class needs no @entity decoration.